## Modular Pipeline Integration Test

In [1]:
# Standard imports
import pandas as pd
import numpy as np
import torch
import os
import json
import sys
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image


# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)


# Pipeline imports
from src.pipeline.data_ingestion import load_csv, build_image_label_df
from src.pipeline.preprocessing import preprocess_data
from src.pipeline.feature_engineering import generate_embeddings
from src.pipeline.datasets import ProductDataset
from src.pipeline.evaluation import evaluate_model
from src.pipeline.inference import predict
from src.initialization import load_environment

# Utility imports
from src.utils.vector_db_utils import VectorDBManager
from src.utils.data_cleaning import clean_stock_code
from src.utils.model_loader import load_model
from src.utils.generate_cnn_csv import generate_final_cnn_training_data

# Script imports
from src.scripts.train_cnn_from_scratch import main

# Model imports
from src.models.cnn_model import CNNModel

# calls
load_environment()
generate_final_cnn_training_data()

INFO:src.utils.model_loader:Loading model from: c:\Users\don\dev\ds_test\models\best_model.pth
INFO:src.utils.model_loader:Loading label mapping from: c:\Users\don\dev\ds_test\src\data\label_mapping.json
INFO:src.utils.model_loader:Loaded label mapping: {'0': 'LUNCH BAG WOODLAND', '1': 'REX CASH+CARRY JUMBO SHOPPER', '2': 'JUMBO STORAGE BAG SUKI', '3': '6 RIBBONS RUSTIC CHARM', '4': 'CHOCOLATE HOT WATER BOTTLE', '5': 'RETROSPOT TEA SET CERAMIC 11 PC', '6': 'LUNCH BAG PINK POLKADOT', '7': 'REGENCY CAKESTAND 3 TIER', '8': 'ALARM CLOCK BAKELIKE RED', '9': 'SPOTTY BUNTING'}
INFO:src.utils.model_loader:Initialized model with 10 classes


[DEBUG] Current working directory: c:\Users\don\dev\ds_test\notebooks
[DEBUG] Model path (as passed): c:\Users\don\dev\ds_test\models\best_model.pth
[DEBUG] Model path (absolute): c:\Users\don\dev\ds_test\models\best_model.pth


INFO:src.utils.model_loader:Loaded state dict with keys: odict_keys(['conv1.0.weight', 'conv1.0.bias', 'conv1.1.weight', 'conv1.1.bias', 'conv1.1.running_mean', 'conv1.1.running_var', 'conv1.1.num_batches_tracked', 'conv2.0.weight', 'conv2.0.bias', 'conv2.1.weight', 'conv2.1.bias', 'conv2.1.running_mean', 'conv2.1.running_var', 'conv2.1.num_batches_tracked', 'conv3.0.weight', 'conv3.0.bias', 'conv3.1.weight', 'conv3.1.bias', 'conv3.1.running_mean', 'conv3.1.running_var', 'conv3.1.num_batches_tracked', 'conv4.0.weight', 'conv4.0.bias', 'conv4.1.weight', 'conv4.1.bias', 'conv4.1.running_mean', 'conv4.1.running_var', 'conv4.1.num_batches_tracked', 'fc.1.weight', 'fc.1.bias', 'fc.4.weight', 'fc.4.bias'])
INFO:src.utils.model_loader:Successfully loaded state dict into model
INFO:src.utils.model_loader:Model set to evaluation mode
INFO:src.utils.model_loader:Test input shape: torch.Size([1, 3, 224, 224])
INFO:src.utils.model_loader:Raw model output shape: torch.Size([1, 10])
INFO:src.utils.m

INFO: Model and transform initialized successfully


INFO:src.initialization:Model and transform initialized successfully


### Data Ingestion & Cleaning
Load raw data from CSV, drop duplicates, apply cleaning/ preprocessing

In [2]:
raw_df = load_csv('../src/data/dataset/dataset.csv')
raw_df = raw_df.drop_duplicates()

# Clean and preprocess
cleaned_df = preprocess_data(raw_df)

cleaned_df = cleaned_df.drop_duplicates(subset='StockCode', keep='last')

cleaned_df.to_csv('../src/data/dataset/cleaned/cleaned_cnn.csv')

c:\Users\don\dev\ds_test\src\utils\data_cleaning.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Quantity'] = df['Quantity'].fillna(1)
c:\Users\don\dev\ds_test\src\utils\data_cleaning.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['InvoiceDate'] = df['InvoiceDate'].fillna('1970-01-01 00:00:00')
c:\Users\don\dev\ds_test\src\utils\data_cleaning.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value in

### Feature Engineering & Embedding Generation
Generate feature embeddings for the deduplicated product data (for vector DB)

In [3]:
api_key = os.getenv('PINECONE_API_KEY')

vector_db_manager = VectorDBManager(api_key=api_key)
vector_db_manager.initialize()

embeddings = generate_embeddings(cleaned_df, vector_db_manager)

embeddings_df = pd.DataFrame(np.array(embeddings))
embeddings_df.head()

INFO: Using existing Pinecone index


INFO:src.utils.vector_db_utils:Using existing Pinecone index
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


INFO: Model loaded


INFO:src.utils.vector_db_utils:Model loaded


INFO: Starting fresh upload


INFO:src.utils.vector_db_utils:Starting fresh upload


Generating embeddings for 3993 rows (batched)...


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Embedding generation complete.


,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,0.044934,0.047733,-0.031713,0.023898,-0.113119,0.020636,0.043290,0.058105,-0.022056,-0.009515,...,-0.107558,0.056651,0.087509,0.049710,0.045068,0.034528,-0.022875,-0.075215,-0.071923,0.016290
1,0.025311,-0.040255,-0.003945,-0.086670,0.041828,-0.073092,0.042201,-0.109977,-0.026584,0.006564,...,0.015641,-0.033279,0.019974,-0.080150,-0.053132,0.086473,0.124811,-0.035266,-0.053563,0.012651
2,0.025311,-0.040255,-0.003945,-0.086670,0.041828,-0.073092,0.042201,-0.109977,-0.026584,0.006564,...,0.015641,-0.033279,0.019974,-0.080150,-0.053132,0.086473,0.124811,-0.035266,-0.053563,0.012651
3,0.006931,-0.003061,-0.006435,0.012643,0.045201,-0.025058,0.055391,-0.064356,-0.039714,-0.002497,...,-0.056848,0.014105,0.026482,-0.008407,0.032257,0.013544,0.025055,-0.103874,-0.052060,0.061078
4,-0.074473,0.044887,-0.013407,-0.087875,-0.098879,0.052754,0.027268,0.017332,-0.102509,-0.055326,...,-0.013475,0.038868,0.049434,0.011181,-0.042980,0.013912,-0.045290,-0.089344,-0.026007,0.092768


### Model Training
Training the model with scraped data

In [10]:
# Run the training script
try:
    history = main(
        project_root_str=project_root,
        batch_size=32,
        num_epochs=50,
        learning_rate=0.001
    )
    display("\nTraining completed successfully!")
    display(f"Best validation accuracy: {history['best_val_acc']:.2f}%")
except Exception as e:
    display(f"Error occurred: {str(e)}")
    raise

INFO: Loaded 437 images for training


INFO:src.scripts.train_cnn_from_scratch:Loaded 437 images for training


INFO: Class distribution:


INFO:src.scripts.train_cnn_from_scratch:Class distribution:


INFO: Class 0: 50 images


...

INFO: After filtering, using 437 images from 10 classes


INFO:src.scripts.train_cnn_from_scratch:After filtering, using 437 images from 10 classes


INFO: Epoch 1/50:


INFO:src.scripts.train_cnn_from_scratch:Epoch 1/50:
Epoch 1/50 [Train]:   0%|          | 0/11 [00:10<?, ?it/s]


KeyboardInterrupt: 

### Evaluation

In [5]:
# Load model after training
model_path = "models/best_model.pth"
label_mapping_path = "src/data/label_mapping.json"
model, label_mapping = load_model(model_path=model_path, label_mapping_path=label_mapping_path)
if model:
    display("Model loaded")

INFO:src.utils.model_loader:Loading model from: c:\Users\don\dev\ds_test\models\best_model.pth
INFO:src.utils.model_loader:Loading label mapping from: c:\Users\don\dev\ds_test\src\data\label_mapping.json
INFO:src.utils.model_loader:Loaded label mapping: {'0': 'LUNCH BAG WOODLAND', '1': 'REX CASH+CARRY JUMBO SHOPPER', '2': 'JUMBO STORAGE BAG SUKI', '3': '6 RIBBONS RUSTIC CHARM', '4': 'CHOCOLATE HOT WATER BOTTLE', '5': 'RETROSPOT TEA SET CERAMIC 11 PC', '6': 'LUNCH BAG PINK POLKADOT', '7': 'REGENCY CAKESTAND 3 TIER', '8': 'ALARM CLOCK BAKELIKE RED', '9': 'SPOTTY BUNTING'}
INFO:src.utils.model_loader:Initialized model with 10 classes


[DEBUG] Current working directory: c:\Users\don\dev\ds_test\notebooks
[DEBUG] Model path (as passed): c:\Users\don\dev\ds_test\models\best_model.pth
[DEBUG] Model path (absolute): c:\Users\don\dev\ds_test\models\best_model.pth


INFO:src.utils.model_loader:Loaded state dict with keys: odict_keys(['conv1.0.weight', 'conv1.0.bias', 'conv1.1.weight', 'conv1.1.bias', 'conv1.1.running_mean', 'conv1.1.running_var', 'conv1.1.num_batches_tracked', 'conv2.0.weight', 'conv2.0.bias', 'conv2.1.weight', 'conv2.1.bias', 'conv2.1.running_mean', 'conv2.1.running_var', 'conv2.1.num_batches_tracked', 'conv3.0.weight', 'conv3.0.bias', 'conv3.1.weight', 'conv3.1.bias', 'conv3.1.running_mean', 'conv3.1.running_var', 'conv3.1.num_batches_tracked', 'conv4.0.weight', 'conv4.0.bias', 'conv4.1.weight', 'conv4.1.bias', 'conv4.1.running_mean', 'conv4.1.running_var', 'conv4.1.num_batches_tracked', 'fc.1.weight', 'fc.1.bias', 'fc.4.weight', 'fc.4.bias'])
INFO:src.utils.model_loader:Successfully loaded state dict into model
INFO:src.utils.model_loader:Model set to evaluation mode
INFO:src.utils.model_loader:Test input shape: torch.Size([1, 3, 224, 224])
INFO:src.utils.model_loader:Raw model output shape: torch.Size([1, 10])
INFO:src.utils.m

'Model loaded'

In [6]:
with open('../models/stockcode_to_index.json') as f:
    stockcode_to_index = json.load(f)

images_root = r'../model_test'
df = build_image_label_df(images_root, stockcode_to_index)

In [7]:
# Prepare the dataset and DataLoader
image_paths = df['image_path']  # List of image paths for each row in train_df
labels = df['label']       # List of class indices for each row in train_df

train_dataset = ProductDataset(image_paths, labels, transform=None)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

# Load the trained model
num_classes = len(set(labels))
model = CNNModel(num_classes)
model.load_state_dict(torch.load('../models/best_model.pth', map_location='cpu'))
model.eval()

# Evaluate
accuracy, metrics = evaluate_model(model, train_loader, labels)
display(f"Training set accuracy: {accuracy}")
display(f"Metrics: {metrics}")

'Training set accuracy: 0.3'

"Metrics: {'accuracy': 0.3}"

### Inference

In [9]:
model_path = '../models/best_model.pth'
num_classes = 10

# Load the model
model = CNNModel(num_classes)
model.load_state_dict(torch.load(model_path, map_location='cpu'))
model.eval()

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load and preprocess the image
img_path = '../model_test/1.jpg'
image = Image.open(img_path).convert('RGB')
input_tensor = transform(image)

# Run inference
predicted_class_idx = predict(model, input_tensor)
display(f"Predicted class index: {predicted_class_idx}")

'Predicted class index: 4'